In [ ]:
import pandas as pd
import google_play_scraper

print("Environment setup successful!")
print("Pandas version:", pd.__version__)

In [ ]:
import pandas as pd
from google_play_scraper import reviews, Sort

In [ ]:
result, continuation_token = reviews(
    'notion.id',
    lang='en',
    country='us',
    sort=Sort.NEWEST,
    count=100
)

print("Number of reviews:", len(result))
print("Continuation token available:", continuation_token is not None)

In [ ]:
df = pd.DataFrame(result)

print("Dataset shape:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nReview content examples:")
print(df[['content', 'score', 'at']].head(10).to_string(index=False))

In [ ]:
next_result, next_token = reviews(
    'notion.id',
    lang='en',
    country='us',
    sort=Sort.NEWEST,
    count=100,
    continuation_token=continuation_token
)

df_next = pd.DataFrame(next_result)

print("Second batch reviews:", len(df_next))
print("Next token available:", next_token is not None)

In [ ]:
duplicate_count = df['reviewId'].isin(
    df_next['reviewId']
).sum()

print("Duplicate reviews:", duplicate_count)

In [ ]:
duolingo_result, duolingo_token = reviews(
    'com.duolingo',
    lang='en',
    country='us',
    sort=Sort.NEWEST,
    count=100
)

df_duolingo = pd.DataFrame(duolingo_result)

print("Number of reviews:", len(df_duolingo))
print("Number of columns:", len(df_duolingo.columns))
print("Continuation token available:", duolingo_token is not None)

In [ ]:
print(df_duolingo.columns.tolist())

print("\nMissing values:")
print(df_duolingo.isnull().sum())

print("\nReview examples:")
print(
    df_duolingo[['content', 'score', 'at']]
    .head(10)
    .to_string(index=False)
)

In [ ]:
duolingo_next_result, duolingo_next_token = reviews(
    'com.duolingo',
    lang='en',
    country='us',
    sort=Sort.NEWEST,
    count=100,
    continuation_token=duolingo_token
)

df_duolingo_next = pd.DataFrame(duolingo_next_result)

print("Second batch reviews:", len(df_duolingo_next))
print("Next token available:", duolingo_next_token is not None)

In [ ]:
duolingo_duplicates = df_duolingo['reviewId'].isin(
    df_duolingo_next['reviewId']
).sum()

print("Duplicate reviews:", duolingo_duplicates)

In [ ]:
spotify_result, spotify_token = reviews(
    'com.spotify.music',
    lang='en',
    country='us',
    sort=Sort.NEWEST,
    count=100
)

df_spotify = pd.DataFrame(spotify_result)

print("Number of reviews:", len(df_spotify))
print("Number of columns:", len(df_spotify.columns))
print("Continuation token available:", spotify_token is not None)

In [ ]:
print(df_spotify.columns.tolist())

print("\nMissing values:")
print(df_spotify.isnull().sum())

print("\nReview examples:")
print(
    df_spotify[['content', 'score', 'at']]
    .head(10)
    .to_string(index=False)
)

In [ ]:
spotify_next_result, spotify_next_token = reviews(
    'com.spotify.music',
    lang='en',
    country='us',
    sort=Sort.NEWEST,
    count=100,
    continuation_token=spotify_token
)

df_spotify_next = pd.DataFrame(spotify_next_result)

print("Second batch reviews:", len(df_spotify_next))
print("Next token available:", spotify_next_token is not None)

In [ ]:
spotify_duplicates = df_spotify['reviewId'].isin(
    df_spotify_next['reviewId']
).sum()

print("Duplicate reviews:", spotify_duplicates)

In [ ]:
from pathlib import Path

data_dir = Path("../data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

notion_all = pd.concat([df, df_next], ignore_index=True)
duolingo_all = pd.concat(
    [df_duolingo, df_duolingo_next],
    ignore_index=True
)
spotify_all = pd.concat(
    [df_spotify, df_spotify_next],
    ignore_index=True
)

notion_all.to_csv(data_dir / "notion_reviews.csv", index=False)
duolingo_all.to_csv(data_dir / "duolingo_reviews.csv", index=False)
spotify_all.to_csv(data_dir / "spotify_reviews.csv", index=False)

print("Data saved successfully!")
print("Notion:", notion_all.shape)
print("Duolingo:", duolingo_all.shape)
print("Spotify:", spotify_all.shape)

In [ ]:
notion_all["app"] = "Notion"
duolingo_all["app"] = "Duolingo"
spotify_all["app"] = "Spotify"

all_reviews = pd.concat(
    [notion_all, duolingo_all, spotify_all],
    ignore_index=True
)

print("Combined dataset shape:", all_reviews.shape)
print("\nReviews by app:")
print(all_reviews["app"].value_counts())

In [ ]:
print("Total rows:", len(all_reviews))

print(
    "Duplicate review IDs:",
    all_reviews.duplicated(
        subset=["app", "reviewId"]
    ).sum()
)

print("\nUnique review IDs by app:")
print(
    all_reviews.groupby("app")["reviewId"].nunique()
)

In [ ]:
all_reviews["content_length"] = (
    all_reviews["content"].astype(str).str.len()
)

print(
    all_reviews.groupby("app")["content_length"]
    .describe()
)

In [ ]:
print(
    "Duplicate review IDs:",
    all_reviews.duplicated(
        subset=["app", "reviewId"]
    ).sum()
)

print("\nUnique review IDs by app:")
print(
    all_reviews.groupby("app")["reviewId"].nunique()
)

In [ ]:
empty_reviews = (
    all_reviews["content"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty review content:", empty_reviews)

In [ ]:
%pip install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

print("VADER initialized successfully!")

In [ ]:
%pip install langdetect

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

def detect_language(text):
    try:
        return detect(str(text))
    except:
        return "unknown"

In [ ]:
all_reviews["language"] = all_reviews["content"].apply(detect_language)

print("Language detection completed!")
print(all_reviews[["app", "content", "language"]].head(10))

In [ ]:
language_summary = (
    all_reviews.groupby(["app", "language"])
    .size()
    .unstack(fill_value=0)
)

print(language_summary)

In [ ]:
language_check = (
    all_reviews.groupby("app")["language"]
    .value_counts()
    .unstack(fill_value=0)
)

summary = pd.DataFrame(index=language_check.index)

summary["total"] = all_reviews.groupby("app").size()
summary["english"] = language_check["en"]
summary["unknown"] = language_check["unknown"]
summary["non_english"] = (
    summary["total"] - summary["english"] - summary["unknown"]
)
summary["english_pct"] = (
    summary["english"] / summary["total"] * 100
).round(2)

print(summary)

In [ ]:
non_english_sample = all_reviews[
    (all_reviews["language"] != "en") &
    (all_reviews["language"] != "unknown")
][["app", "content", "language", "content_length"]]

print(non_english_sample.head(20).to_string(index=False))

In [ ]:
suspected_reviews = all_reviews[
    (all_reviews["language"] != "en") &
    (all_reviews["content_length"] <= 20)
][["app", "content", "language", "content_length"]]

print(suspected_reviews.to_string(index=False))

In [ ]:
english_phrases = [
    "not good", "excellent", "great nice", "best app ever",
    "let's hope", "good", "gooo", "i love you notion",
    "useless", "super", "ai is paid", "very good",
    "amazing", "best for notes", "verry good app",
    "useful app", "fantastic", "best app forever",
    "wonderful", "so good", "good app", "super cool",
    "very happy", "i love it", "dope", "nice app",
    "so interesting", "super awesome", "great app",
    "perfect", "very nice", "great lessons", "good one",
    "i love music", "best app", "not so good",
    "nice music", "fabulous", "awesome", "the best",
    "excellent!!!", "very satisfied", "amazing music app"
]

normalized_content = (
    all_reviews["content"]
    .str.lower()
    .str.strip()
)

all_reviews["language_final"] = all_reviews["language"]

all_reviews.loc[
    normalized_content.isin(english_phrases),
    "language_final"
] = "en"

print(
    all_reviews["language_final"]
    .value_counts()
    .head(15)
)

In [ ]:
english_reviews = all_reviews[
    all_reviews["language_final"] == "en"
].copy()

print("English reviews:", len(english_reviews))
print("\nReviews by app:")
print(english_reviews["app"].value_counts())

print("\nSample reviews:")
print(english_reviews[["app", "content", "score"]].head(10))


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

sentiment_scores = english_reviews["content"].apply(
    analyzer.polarity_scores
)

sentiment_df = pd.DataFrame(sentiment_scores.tolist())

english_reviews = pd.concat(
    [english_reviews.reset_index(drop=True),
     sentiment_df.reset_index(drop=True)],
    axis=1
)

print(
    english_reviews[
        ["app", "content", "score", "pos", "neu", "neg", "compound"]
    ].head(10)
)

In [ ]:
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

english_reviews["sentiment"] = (
    english_reviews["compound"].apply(classify_sentiment)
)

print(english_reviews[["app", "content", "compound", "sentiment"]].head(10))

In [ ]:
sentiment_summary = pd.crosstab(
    english_reviews["app"],
    english_reviews["sentiment"],
    normalize="index"
) * 100

print(sentiment_summary.round(2))

In [ ]:
def classify_rating(score):
    if score <= 2:
        return "Negative"
    elif score == 3:
        return "Neutral"
    else:
        return "Positive"

english_reviews["rating_sentiment"] = (
    english_reviews["score"].apply(classify_rating)
)

In [ ]:
comparison = pd.crosstab(
    english_reviews["rating_sentiment"],
    english_reviews["sentiment"],
    margins=True
)

print(comparison)

In [ ]:
mismatch_reviews = english_reviews[
    english_reviews["rating_sentiment"] != english_reviews["sentiment"]
].copy()

print("Number of mismatched reviews:", len(mismatch_reviews))

print(
    mismatch_reviews[
        ["app", "content", "score", "compound",
         "rating_sentiment", "sentiment"]
    ].head(20).to_string(index=False)
)

In [ ]:
mismatch_reviews["mismatch_type"] = (
    mismatch_reviews["rating_sentiment"]
    + " rating / "
    + mismatch_reviews["sentiment"]
    + " text"
)

mismatch_summary = pd.crosstab(
    mismatch_reviews["app"],
    mismatch_reviews["mismatch_type"]
)

print(mismatch_summary)
print("\nTotal mismatches by app:")
print(mismatch_summary.sum(axis=1))

In [ ]:
issue_keywords = {
    "Performance": [
        "slow", "lag", "laggy", "freeze", "freezing",
        "crash", "crashes", "crashing", "loading"
    ],
    "Login and Access": [
        "login", "log in", "sign in", "sign-in",
        "password", "account", "locked out"
    ],
    "Pricing": [
        "price", "pricing", "expensive", "subscription",
        "paid", "payment", "cost", "fee"
    ],
    "Features and Functionality": [
        "feature", "function", "update", "missing",
        "sync", "integration", "notification"
    ],
    "Usability": [
        "confusing", "difficult", "hard to use",
        "complicated", "user friendly", "interface"
    ]
}

In [ ]:
negative_reviews = english_reviews[
    english_reviews["sentiment"] == "Negative"
].copy()

def identify_issues(text):
    text = str(text).lower()
    matched_issues = []

    for issue, keywords in issue_keywords.items():
        if any(keyword in text for keyword in keywords):
            matched_issues.append(issue)

    return matched_issues

negative_reviews["issues"] = (
    negative_reviews["content"].apply(identify_issues)
)

print("Total negative reviews:", len(negative_reviews))

print(
    negative_reviews[
        ["app", "content", "sentiment", "issues"]
    ].head(15).to_string(index=False)
)

In [ ]:
issue_exploded = negative_reviews[
    ["app", "issues"]
].copy()

issue_exploded = issue_exploded.explode("issues").reset_index(drop=True)

issue_exploded = issue_exploded.dropna(subset=["issues"])

issue_summary = (
    issue_exploded
    .groupby(["app", "issues"])
    .size()
    .unstack(fill_value=0)
)

print(issue_summary)

In [ ]:
issue_percentage = (
    issue_summary.div(issue_summary.sum(axis=1), axis=0) * 100
).round(2)

print(issue_percentage)

In [ ]:
unclassified = negative_reviews[
    negative_reviews["issues"].apply(len) == 0
]

print("Unclassified negative reviews:", len(unclassified))

print(
    unclassified[["app", "content"]]
    .head(10)
    .to_string(index=False)
)

In [ ]:
issue_keywords = {
    "Performance": [
        "slow", "lag", "laggy", "freeze", "freezing",
        "crash", "crashes", "crashing", "loading",
        "buggy", "unresponsive", "stuck"
    ],
    "Login and Access": [
        "login", "log in", "logging in", "sign in",
        "sign-in", "signing in", "password",
        "account", "locked out", "sign up", "signing up"
    ],
    "Pricing": [
        "price", "pricing", "expensive", "subscription",
        "paid", "payment", "cost", "fee", "overpriced",
        "too expensive"
    ],
    "Features and Functionality": [
        "feature", "function", "update", "missing",
        "sync", "integration", "notification",
        "text size", "line break", "search",
        "find and replace", "autocorrect"
    ],
    "Usability": [
        "confusing", "difficult", "hard to use",
        "complicated", "user friendly", "interface",
        "annoying", "frustrating"
    ],
    "Data Loss and Reliability": [
        "data loss", "lost data", "deleted",
        "missing notes", "lost notes", "recover",
        "recovery", "cannot be trusted", "unsafe"
    ],
    "Bugs and Errors": [
        "bug", "bugs", "buggy", "glitch",
        "glitching", "not working", "stopped working",
        "doesn't work", "does not work", "error",
        "broken", "malfunction"
    ]
}

In [ ]:
negative_reviews["issues"] = negative_reviews["content"].apply(
    identify_issues
)

In [ ]:
unclassified = negative_reviews[
    negative_reviews["issues"].apply(len) == 0
]

print("Unclassified negative reviews:", len(unclassified))
print(
    unclassified[["app", "content"]]
    .head(15)
    .to_string(index=False)
)

In [ ]:
print(
    unclassified[["app", "content"]]
    .to_string(index=False)
)

In [ ]:
sentiment_check = negative_reviews[
    negative_reviews["content"].str.contains(
        "10/10|only music app|fix my mood|fun as hell",
        case=False,
        na=False
    )
]

print(
    sentiment_check[
        ["app", "content", "score", "compound", "sentiment"]
    ].to_string(index=False)
)

In [ ]:
english_reviews["sentiment_final"] = english_reviews["sentiment"]

english_reviews["validation_method"] = "VADER"

manual_corrections = {
    "honestly fun as hell I don't even care about learning languages anymore": "Positive",
    "10/10 the only music app ill ever use": "Positive",
    "It become my part of life and always fix my mood when i am sad ...": "Positive"
}

for review, corrected_sentiment in manual_corrections.items():
    mask = english_reviews["content"] == review
    english_reviews.loc[mask, "sentiment_final"] = corrected_sentiment
    english_reviews.loc[mask, "validation_method"] = "Manual correction"

In [ ]:
print(
    english_reviews[
        english_reviews["validation_method"] == "Manual correction"
    ][
        ["app", "content", "sentiment", "sentiment_final", "validation_method"]
    ].to_string(index=False)
)

In [ ]:
print("Original VADER sentiment:")
print(
    english_reviews["sentiment"]
    .value_counts()
    .sort_index()
)

print("\nFinal sentiment after manual corrections:")
print(
    english_reviews["sentiment_final"]
    .value_counts()
    .sort_index()
)

In [ ]:
original_distribution = pd.crosstab(
    english_reviews["app"],
    english_reviews["sentiment"],
    normalize="index"
) * 100

final_distribution = pd.crosstab(
    english_reviews["app"],
    english_reviews["sentiment_final"],
    normalize="index"
) * 100

print("Original distribution:")
print(original_distribution.round(2))

print("\nFinal distribution:")
print(final_distribution.round(2))

In [ ]:
negative_reviews_final = english_reviews[
    english_reviews["sentiment_final"] == "Negative"
].copy()

print("Original negative reviews:", len(negative_reviews))
print("Final negative reviews:", len(negative_reviews_final))

print("\nNegative reviews by app:")
print(negative_reviews_final["app"].value_counts())

In [ ]:
negative_reviews_final["issues"] = (
    negative_reviews_final["content"].apply(identify_issues)
)

issue_exploded_final = (
    negative_reviews_final[["app", "issues"]]
    .copy()
    .explode("issues")
    .reset_index(drop=True)
)

issue_exploded_final = issue_exploded_final.dropna(
    subset=["issues"]
)

issue_summary_final = (
    issue_exploded_final
    .groupby(["app", "issues"])
    .size()
    .unstack(fill_value=0)
)

print(issue_summary_final)

In [ ]:
issue_percentage_final = (
    issue_summary_final.div(
        issue_summary_final.sum(axis=1), axis=0
    ) * 100
).round(2)

print(issue_percentage_final)

In [ ]:
unclassified_final = negative_reviews_final[
    negative_reviews_final["issues"].apply(len) == 0
]

print("Unclassified negative reviews:", len(unclassified_final))

print(
    unclassified_final[["app", "content"]]
    .to_string(index=False)
)

In [ ]:
print("Total negative reviews by app:")
print(negative_reviews_final["app"].value_counts())

print("\nClassified reviews by app:")
print(
    negative_reviews_final[
        negative_reviews_final["issues"].apply(len) > 0
    ]["app"].value_counts()
)

In [ ]:
unclassified_final = negative_reviews_final[
    negative_reviews_final["issues"].apply(len) == 0
]

print(
    unclassified_final[["app", "content"]]
    .to_string(index=False)
)

In [ ]:
issue_keywords = {
    "Performance": [
        "slow", "lag", "laggy", "freeze", "freezing",
        "crash", "crashes", "crashing", "loading",
        "buggy", "unresponsive", "stuck",
        "cpu", "battery", "heats up"
    ],

    "Login and Access": [
        "login", "log in", "logging in", "sign in",
        "sign-in", "signing in", "password",
        "account", "locked out", "sign up", "signing up"
    ],

    "Pricing": [
        "price", "pricing", "expensive", "subscription",
        "paid", "payment", "cost", "fee", "overpriced",
        "too expensive", "pay", "paying", "premium",
        "free version", "free options", "yearly"
    ],

    "Features and Functionality": [
        "feature", "function", "update", "missing",
        "sync", "integration", "notification",
        "text size", "line break", "search",
        "find and replace", "autocorrect",
        "copy", "widget", "voice recognition",
        "calibration", "rating system"
    ],

    "Usability": [
        "confusing", "difficult", "hard to use",
        "complicated", "user friendly", "interface",
        "annoying", "frustrating", "too many tabs",
        "too many screens", "click through",
        "learning curve", "easy to use"
    ],

    "Data Loss and Reliability": [
        "data loss", "lost data", "deleted",
        "missing notes", "lost notes", "recover",
        "recovery", "cannot be trusted", "unsafe"
    ],

    "Bugs and Errors": [
        "bug", "bugs", "buggy", "glitch",
        "glitching", "not working", "stopped working",
        "doesn't work", "does not work", "error",
        "broken", "malfunction"
    ],

    "Advertising": [
        "ads", "advertisement", "advertisements",
        "bombarded with ads", "too many ads",
        "watching ads", "ad-free"
    ],

    "Content Quality": [
        "content quality", "translation", "inaccurate",
        "wrong translation", "wrong answer",
        "course quality", "curriculum", "lesson",
        "lessons", "learning", "teach",
        "teaching", "nonsense", "unusable",
        "ai slop", "voice recognition"
    ],

    "Recommendations": [
        "algorithm", "recommendation", "recommendations",
        "relevant selections", "irrelevant",
        "not relevant", "suggestions"
    ],

    "Playback Experience": [
        "shuffle", "repeating the song", "repeat the song",
        "skips", "skipping", "autoplay",
        "starts playing music", "song", "songs"
    ],

    "Product Design": [
        "icon", "design", "dark patterns",
        "product direction", "ai forward",
        "too many tabs", "vibrations",
        "character", "product managers"
    ]
}

In [ ]:
negative_reviews_final["issues"] = (
    negative_reviews_final["content"].apply(identify_issues)
)

In [ ]:
issue_exploded_final = (
    negative_reviews_final[["app", "issues"]]
    .explode("issues")
    .dropna(subset=["issues"])
    .reset_index(drop=True)
)

issue_summary_final = (
    issue_exploded_final
    .groupby(["app", "issues"])
    .size()
    .unstack(fill_value=0)
)

print(issue_summary_final)

In [ ]:
print(
    negative_reviews_final["issues"]
    .apply(len)
    .gt(0)
    .groupby(negative_reviews_final["app"])
    .agg(["sum", "count"])
)

In [ ]:
multi_label_reviews = negative_reviews_final[
    negative_reviews_final["issues"].apply(len) > 1
]

print(
    multi_label_reviews[["app", "content", "issues"]]
    .to_string(index=False)
)

In [ ]:
import re

def identify_issues(text):
    text = str(text).lower()
    matched_issues = []

    for issue, keywords in issue_keywords.items():
        for keyword in keywords:
            pattern = r"(?<!\w)" + re.escape(keyword) + r"(?!\w)"

            if re.search(pattern, text):
                matched_issues.append(issue)
                break

    return matched_issues

In [ ]:
negative_reviews_final["issues"] = (
    negative_reviews_final["content"].apply(identify_issues)
)

In [ ]:
coverage = (
    negative_reviews_final["issues"]
    .apply(len)
    .gt(0)
    .groupby(negative_reviews_final["app"])
    .agg(["sum", "count"])
)

coverage["coverage_rate"] = (
    coverage["sum"] / coverage["count"] * 100
).round(2)

print(coverage)

In [ ]:
unclassified_final = negative_reviews_final[
    negative_reviews_final["issues"].apply(len) == 0
]

print(
    unclassified_final[["app", "content"]]
    .to_string(index=False)
)

In [ ]:
manual_issue_corrections = {
    "Mobile version is terrible. whenever something autocorrects, the cursor jumps back to where the correction was in the word🤦‍♂️": [
        "Features and Functionality"
    ],

    "Too many fancy features to the extent it is hard to do straightforward things I want to do then rather than have AI forced on me instead of letting me choose.": [
        "Usability",
        "Product Design"
    ],

    "2023: They really managed to make even copying a part of text a pain. Saving notes a pain, moving notes a pain. Can't handle anymore. Going back to Obsidian. 2026: Trying it again I remembered this a bs app. You need to study it for months to be able to use it. Wanted to use it to journal but I cant't.": [
        "Features and Functionality",
        "Usability"
    ],

    "This app helped me learn a bit of french. but the thing makes you dismiss more important things and it is just a flop not good.": [
        "Product Design"
    ],

    "This always gets \"or\" for Spanish wrong no matter how I say \"o\"": [
        "Content Quality"
    ],

    "Rare Scam. There is a daily limit per day and if I play more in chess and win, I will get 0 elo, but if I lose - I will lose elo.": [
        "Features and Functionality"
    ],

    "Meh, I found this not good. Like, when I didn't do a language until night, the widgets get worse!": [
        "Features and Functionality"
    ],

    "don't like the name": [
        "Product Design"
    ],

    "Imperfect but Good. Why is there No Edit button? I often comment then spell check steps in and corrects something in a stupid manner. Then I must either leave the comment or delete and resubmit it. Ridiculous. Otherwise the content works well and the App functions well.": [
        "Features and Functionality"
    ]
}

for review, issues in manual_issue_corrections.items():
    mask = negative_reviews_final["content"] == review
    negative_reviews_final.loc[mask, "issues"] = (
        negative_reviews_final.loc[mask, "issues"]
        .apply(lambda x: list(set(x + issues)))
    )

In [ ]:
issue_exploded_final = (
    negative_reviews_final[["app", "issues"]]
    .explode("issues")
    .dropna(subset=["issues"])
)

issue_summary_final = (
    issue_exploded_final
    .groupby(["app", "issues"])
    .size()
    .unstack(fill_value=0)
)

issue_prevalence = (
    issue_summary_final
    .div(negative_reviews_final["app"].value_counts(), axis=0)
    * 100
).round(2)

print("Issue counts:")
print(issue_summary_final)

print("\nIssue prevalence (% of all negative reviews):")
print(issue_prevalence)

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Define the issue order
issue_order = [
    "Advertising",
    "Bugs and Errors",
    "Content Quality",
    "Data Loss and Reliability",
    "Features and Functionality",
    "Login and Access",
    "Performance",
    "Playback Experience",
    "Pricing",
    "Product Design",
    "Recommendations",
    "Usability"
]

# Arrange the data
heatmap_data = issue_prevalence.reindex(
    columns=issue_order,
    fill_value=0
)

# Create the heatmap
fig, ax = plt.subplots(figsize=(16, 5))

im = ax.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=60
)

# Add labels
ax.set_xticks(range(len(issue_order)))
ax.set_xticklabels(issue_order, rotation=45, ha="right")

ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

# Add percentage values
for i in range(len(heatmap_data.index)):
    for j in range(len(issue_order)):
        value = heatmap_data.iloc[i, j]
        ax.text(
            j, i, f"{value:.1f}%",
            ha="center",
            va="center",
            fontsize=8
        )

ax.set_title(
    "Negative Review Issue Prevalence by Application",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Product Issue Category")
ax.set_ylabel("Application")

fig.colorbar(
    im,
    ax=ax,
    label="Percentage of Negative Reviews (%)"
)

plt.tight_layout()
plt.savefig(
    "../data/issue_prevalence_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:

# Select analysis-related columns
analysis_columns = [
    "app",
    "score",
    "at",
    "appVersion",
    "language_final",
    "pos",
    "neu",
    "neg",
    "compound",
    "sentiment",
    "sentiment_final",
    "validation_method",
    "rating_sentiment"
]

analysis_dataset = english_reviews[
    analysis_columns
].copy()

# Add issue labels to negative reviews
analysis_dataset["issues"] = None

analysis_dataset.loc[
    negative_reviews_final.index,
    "issues"
] = negative_reviews_final["issues"]

# Save the cleaned dataset
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

analysis_dataset.to_csv(
    output_dir / "english_reviews_analysis.csv",
    index=False
)

print("Dataset shape:", analysis_dataset.shape)
print("\nColumns:")
print(analysis_dataset.columns.tolist())

print("\nSentiment distribution:")
print(analysis_dataset["sentiment_final"].value_counts())


In [ ]:

print("Dataset shape:", analysis_dataset.shape)

print("\nMissing values:")
print(analysis_dataset.isnull().sum())

print("\nReviews by app:")
print(analysis_dataset["app"].value_counts())

print("\nIssue labels example:")
print(
    analysis_dataset[
        analysis_dataset["issues"].notna()
    ][["app", "sentiment_final", "issues"]].head(10)
)


In [ ]:
analysis_dataset.to_csv(
    output_dir / "english_reviews_analysis.csv",
    index=False
)

In [ ]:
negative_data = analysis_dataset[
    analysis_dataset["sentiment_final"] == "Negative"
]

labeled_count = negative_data["issues"].apply(len).gt(0).sum()
unlabeled_count = len(negative_data) - labeled_count

print("Total negative reviews:", len(negative_data))
print("Negative reviews with issue labels:", labeled_count)
print("Negative reviews without issue labels:", unlabeled_count)
print("Issue labeling coverage:", f"{labeled_count / len(negative_data) * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

issue_order = [
    "Advertising",
    "Bugs and Errors",
    "Content Quality",
    "Data Loss and Reliability",
    "Features and Functionality",
    "Login and Access",
    "Performance",
    "Playback Experience",
    "Pricing",
    "Product Design",
    "Recommendations",
    "Usability"
]

heatmap_data = issue_prevalence.reindex(
    columns=issue_order,
    fill_value=0
)

fig, ax = plt.subplots(figsize=(16, 5))

im = ax.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=60
)

ax.set_xticks(range(len(issue_order)))
ax.set_xticklabels(issue_order, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

for i in range(len(heatmap_data.index)):
    for j in range(len(issue_order)):
        value = heatmap_data.iloc[i, j]
        ax.text(
            j, i, f"{value:.1f}%",
            ha="center",
            va="center",
            fontsize=8
        )

ax.set_title("Negative Review Issue Prevalence by Application")
ax.set_xlabel("Product Issue Category")
ax.set_ylabel("Application")

fig.colorbar(
    im,
    ax=ax,
    label="Percentage of Negative Reviews (%)"
)

plt.tight_layout()

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "issue_prevalence_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
